# Survival Analysis RAG — Walkthrough

This notebook walks through the full RAG pipeline for survival analysis self-study. By the end you'll have:

- A local ChromaDB index of your textbook and the lung cancer dataset codebook
- A working `query.py` that returns Socratic responses grounded in retrieved passages
- An understanding of why two-pronged retrieval (book vs dataset) matters

**You don't need the PDF to run most of this notebook.** Sections 1–5 work with the dataset codebook alone. Section 6 adds textbook retrieval once you have the PDF.

---

## What RAG means here

RAG (Retrieval-Augmented Generation) means: instead of asking a language model a question cold, you first retrieve relevant passages from your own materials, then pass those passages as context to the model.

The difference matters because:
- The model's answer is grounded in *your* textbook, not its training data
- You can see exactly which passages informed the response
- The Socratic tutor uses retrieved passages as questions to ask, not answers to give back

This turns your PDF into something queryable — a research assistant that has actually read your book.

## Section 1 — Install dependencies

In [ ]:
# Run this cell once to install requirements
# (If you're in a venv or conda env, install from the terminal instead)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', '../requirements.txt', '-q'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else 'Installed.')
if result.returncode != 0:
    print('ERRORS:', result.stderr[-500:])

In [ ]:
# Set your OpenAI API key
# Option A: set in environment before starting Jupyter (recommended)
# Option B: paste it here temporarily (never commit this cell)

import os
from pathlib import Path

# Try to load from .env file if it exists
env_path = Path('..') / '.env'
if env_path.exists():
    from dotenv import load_dotenv
    load_dotenv(env_path)
    print('Loaded API key from .env')

if not os.environ.get('OPENAI_API_KEY'):
    print('WARNING: OPENAI_API_KEY not set. Embedding and query calls will fail.')
    print('Set it with: os.environ["OPENAI_API_KEY"] = "sk-..."')
else:
    key = os.environ['OPENAI_API_KEY']
    print(f'API key found: {key[:8]}...')

## Section 2 — Download and explore the lung dataset

In [ ]:
# Download the lung dataset if not already present
from pathlib import Path
import subprocess, sys

lung_path = Path('../data/lung.csv')
if lung_path.exists():
    print(f'lung.csv already exists at {lung_path.resolve()}')
else:
    print('Downloading lung dataset...')
    result = subprocess.run(
        [sys.executable, '../data/get_lung.py'],
        capture_output=True, text=True, cwd='..'
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/lung.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Understand the outcome variable before anything else
print('Censoring status breakdown:')
print(df['status'].value_counts())
print('\n  1 = censored (alive at end of study)')
print('  2 = dead (event occurred)')
print(f'\nEvent rate: {(df["status"] == 2).mean():.1%} of patients died during follow-up')
print(f'Median follow-up: {df["time"].median():.0f} days ({df["time"].median()/30:.1f} months)')

In [ ]:
# Missing data — critical to understand before modeling
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Missing N': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing N'] > 0]

In [ ]:
# Quick KM survival curve using lifelines
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall KM
kmf.fit(df['time'], event_observed=(df['status'] == 2))
kmf.plot_survival_function(ax=axes[0], ci_show=True)
axes[0].set_title('Overall Survival — NCCTG Lung Cancer')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Survival probability')

# Stratified by sex
for sex, label in [(1, 'Male'), (2, 'Female')]:
    mask = df['sex'] == sex
    kmf.fit(df.loc[mask, 'time'], event_observed=(df.loc[mask, 'status'] == 2), label=label)
    kmf.plot_survival_function(ax=axes[1], ci_show=False)
axes[1].set_title('Survival by Sex')
axes[1].set_xlabel('Days')

plt.tight_layout()
plt.show()
print('Median survival (overall):', kmf.median_survival_time_)

## Section 3 — Build the ChromaDB index

In [ ]:
# Ingest the dataset codebook (no PDF needed for this step)
# This creates chroma_db/ in the examples/survival-rag/ directory
import subprocess, sys

result = subprocess.run(
    [sys.executable, '../ingest.py', '--dataset-only', '--lung-csv', '../data/lung.csv'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

In [ ]:
# If you have the textbook PDF, ingest it here
# Replace the path below with your actual PDF location

# Uncomment and run:
# pdf_path = Path.home() / 'books' / 'klein_moeschberger.pdf'
# result = subprocess.run(
#     [sys.executable, '../ingest.py', '--pdf-path', str(pdf_path), '--lung-csv', '../data/lung.csv'],
#     capture_output=True, text=True, cwd='..'
# )
# print(result.stdout)

print('PDF ingestion step (commented out — run after you have the textbook)')

In [ ]:
# Verify what's in ChromaDB
import sys
sys.path.insert(0, '..')
import chromadb

db = chromadb.PersistentClient(path='../chroma_db')
for col in db.list_collections():
    print(f'Collection: {col.name} — {col.count()} documents')

## Section 4 — Ask three sample questions

Three question types that exercise different parts of the pipeline:
1. **Conceptual** — `"What is the hazard function?"` → should retrieve textbook passages (or reason from general knowledge)
2. **Dataset variable** — `"What does the status variable mean?"` → should retrieve the codebook entry for `status`
3. **Applied** — `"Why does sex matter in this dataset?"` → should retrieve both codebook stats and any textbook discussion of covariates

Watch how the tutor responds differently to each type.

In [ ]:
# Helper: call query.py and capture output
import subprocess, sys

def ask(question: str, verbose: bool = False) -> str:
    cmd = [sys.executable, '../query.py']
    if verbose:
        cmd.append('--verbose')
    cmd.append(question)
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='..')
    if result.returncode != 0:
        return f'ERROR: {result.stderr[:300]}'
    return result.stdout

In [ ]:
# Question 1: Conceptual — hazard function
# If you have the textbook ingested, this will retrieve relevant passages.
# Without the textbook, the tutor responds from general knowledge.

print(ask('What is the hazard function?', verbose=True))

In [ ]:
# Question 2: Dataset variable — status variable
# This should always work since we ingested the dataset codebook.
# Watch for 'From the dataset codebook...' in the response.

print(ask('What does the status variable mean?', verbose=True))

In [ ]:
# Question 3: Applied — interpreting sex in this dataset
# Should retrieve codebook entry for sex + possibly textbook on covariates.
# Notice how the tutor connects the codebook description to a guiding question.

print(ask('Why does sex matter in the lung cancer survival analysis?', verbose=True))

## Section 5 — Compare: with vs without retrieved context

The key question for any RAG system: does retrieval actually help?

Here we query directly against the LLM (no retrieval) and compare it to the RAG response. The comparison shows:
- Whether the RAG response is grounded in your materials vs. generic training data
- Whether source attribution actually shows up in the response
- Whether the Socratic framing differs when the LLM has concrete context

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

QUESTION = 'What does the status variable mean in the lung dataset?'

# WITHOUT retrieval — plain LLM response
no_rag_resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': 'You are a helpful data science tutor.'},
        {'role': 'user', 'content': QUESTION},
    ],
    max_tokens=300,
)

print('=== WITHOUT RAG ===')
print(no_rag_resp.choices[0].message.content)
print()

In [ ]:
# WITH retrieval — RAG response from query.py
print('=== WITH RAG ===')
print(ask(QUESTION, verbose=True))

In [ ]:
# What to notice in the comparison:
# 1. WITHOUT RAG: the LLM may give a generic survival analysis answer
#    (e.g., "1 = censored, 2 = dead" which is correct but not specific to this dataset)
# 2. WITH RAG: the response draws on the codebook entry, which includes
#    dataset-specific details (28% censoring rate, study design, coding note)
#    and is framed as a question to guide understanding, not just facts

print('Observations to note:')
print('  - Did the RAG response mention specifics from the codebook?')
print('  - Did the Socratic framing differ between the two?')
print('  - Would either response mislead a student?')

## Section 6 — Adding the textbook PDF

To get textbook retrieval working:

1. Obtain a PDF of your survival analysis textbook:
   - **Klein & Moeschberger**: *Survival Analysis: Techniques for Censored and Truncated Data* (2nd ed., Springer 2003) — the canonical applied textbook
   - **Kalbfleisch & Prentice**: *The Statistical Analysis of Failure Time Data* (2nd ed., Wiley 2002) — more theoretical
   - Any other PDF textbook works — the pipeline is book-agnostic

2. Run ingest with the PDF:
```bash
python ingest.py --pdf-path ~/books/klein_moeschberger.pdf --lung-csv data/lung.csv
```

3. Re-run the query cells above. You'll see `[Textbook passage N — page X]` appear in verbose mode, and responses will include phrases like "The textbook introduces this as..."

The pipeline is also book-agnostic — it works with any survival analysis PDF:
- Your course lecture notes as PDF
- A paper on tied events in Cox models
- A chapter from Harrell's *Regression Modeling Strategies*

Each new PDF ingested is added to the same `textbook_chunks` collection. Chunks from different sources are distinguished by `pdf_name` in the metadata.

In [ ]:
# After ingesting the textbook, try these three queries to see retrieval in action:
questions = [
    'What is the hazard function and how does it relate to survival probability?',
    'How do I interpret a hazard ratio from a Cox model?',
    'When should I use the log-rank test vs the Cox model?',
]
for q in questions:
    print(f'\n>>> {q}')
    print(ask(q, verbose=True))

## Section 7 — Extending to your own materials

This pipeline is a template. Here's how to adapt it:

**Different dataset:**
- Edit the `LUNG_CODEBOOK` dict in `ingest.py` with your dataset's variable descriptions
- Add a `DATASET_OVERVIEW` and `DATASET_MODELING_NOTES` block for your dataset
- Run `ingest.py --dataset-only`

**Different textbook:**
- Point `--pdf-path` at any PDF — the chunking and embedding are book-agnostic
- If your PDF has poor text extraction (scanned), consider running OCR first with `ocrmypdf`

**Different embedding model:**
- See the Ollama alternative in `ingest.py` for local embeddings with `nomic-embed-text`
- Delete `chroma_db/` when switching models — embeddings are dimension-specific

**Different Socratic style:**
- Edit `SOCRATIC_SYSTEM_PROMPT` in `query.py` to change the tutor's behavior
- Lower `temperature` (0.2) for more consistent responses; higher (0.7) for more variation
- Increase `N_RESULTS` to retrieve more passages (may dilute relevance)

**Integration with the Codex tutor:**
- Run `query.py` from within a Codex session to ground responses in retrieved context
- The output format (source attribution + Socratic question) is designed to be readable by the tutor overlay
- See `skills/overlays/student/socratic-tutor/SKILL.md` for the interaction model

In [ ]:
# Final check: what's in your ChromaDB?
import chromadb
db = chromadb.PersistentClient(path='../chroma_db')
print('ChromaDB collections:')
for col in db.list_collections():
    print(f'  {col.name}: {col.count()} documents')